In [ ]:
import os, csv, re
from pathlib import Path
from typing import Dict, Iterable, Set, List, Tuple
from collections import Counter, defaultdict

In [ ]:
# Configurable GTFS data folder:
# 1) Use GTFS_DATA_DIR if defined.
# 2) Otherwise default to .src/gtfs/data from the current working directory.
_default_data_dir = Path.cwd() / ".src" / "gtfs" / "data"
BASE = str(Path(os.environ.get("GTFS_DATA_DIR", str(_default_data_dir))).resolve())

PATHWAYS_FILE = os.path.join(BASE, "pathways.txt")
TRANSFERS_FILE = os.path.join(BASE, "transfers.txt")
STOPS_FILE = os.path.join(BASE, "stops.txt")
STOP_TIMES_FILE = os.path.join(BASE, "stop_times.txt")
STOP_TIMES_CLEANED_FILE = os.path.join(BASE, "stop_times_cleaned.txt")
TRIPS_FILE = os.path.join(BASE, "trips.txt")
ROUTES_FILE = os.path.join(BASE, "routes.txt")

In [ ]:
def check_missing_files(list_of_files: List[str]) -> None:
    '''Prints files that were not found.'''
    missing_files = [p for p in list_of_files if not os.path.exists(p)]
    if missing_files:
        print("The following files were not found:")
        for p in missing_files:
            print(" -", p)
    return

In [ ]:
def sniff_dialect(file_path: str) -> csv.Dialect:
    '''Automatically detects the CSV separator and returns a dialect; falls back to comma by default.'''
    try:
        with open(file_path, 'r', encoding='utf-8-sig', newline='') as f:
            sample = f.read(65536)
        return csv.Sniffer().sniff(sample, delimiters=",;\t")
    except Exception:
        class _D(csv.Dialect):
            delimiter = ','
            quotechar = '"'
            doublequote = True
            skipinitialspace = False
            lineterminator = '\n'
            quoting = csv.QUOTE_MINIMAL
        return _D

In [ ]:
def read_dict_rows(file_path: str) -> Iterable[Dict[str, str]]:
    '''Reads the CSV (auto dialect) and returns rows as dicts with lowercase keys and stripped values.'''
    dialect = sniff_dialect(file_path)
    with open(file_path, 'r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f, dialect=dialect)
        if reader.fieldnames is None:
            raise RuntimeError(f"The file {os.path.basename(file_path)} has no header.")
        fn_map = {name: name.lower().strip() for name in reader.fieldnames}
        for row in reader:
            out = {}
            for k, v in row.items():
                lk = fn_map.get(k, k).lower()
                sv = (v if v is not None else '').strip()
                out[lk] = sv
            yield out

In [ ]:
def get_first_nonempty(row: Dict[str, str], *names: str) -> str:
    '''Returns the first non-empty value from the given field names.'''
    for n in names:
        v = row.get(n, '').strip()
        if v:
            return v
    return ''

In [ ]:
def load_stop_ids(file_path: str) -> List[str]:
    '''Loads all stop_id from the file as a list.'''
    ids: List[str] = []
    for r in read_dict_rows(file_path):
        sid = r.get('stop_id', '').strip()
        if sid:
            ids.append(sid)
    return ids

In [ ]:
def load_stop_names(file_path: str) -> Dict[str, str]:
    """Returns a map of stop_id -> stop_name."""
    names: Dict[str, str] = {}
    for r in read_dict_rows(file_path):
        sid = r.get('stop_id', '').strip()
        sname = r.get('stop_name', '').strip()
        if sid:
            names[sid] = sname
    return names

In [ ]:
def load_pathway_ids(file_path: str) -> Set[str]:
    '''Returns the set of pathway_id from the file.'''
    ids: Set[str] = set()
    for r in read_dict_rows(file_path):
        pid = r.get('pathway_id', '').strip()
        if pid:
            ids.add(pid)
    return ids

In [ ]:
def load_route_ids(file_path: str) -> Set[str]:
    '''Returns the set of route_id.'''
    ids: Set[str] = set()
    for r in read_dict_rows(file_path):
        rid = r.get('route_id', '').strip()
        if rid:
            ids.add(rid)
    return ids

In [ ]:
def load_trip_ids(file_path: str) -> Set[str]:
    '''Returns the set of trip_id from the file.'''
    ids: Set[str] = set()
    for r in read_dict_rows(file_path):
        tid = r.get('trip_id', '').strip()
        if tid:
            ids.add(tid)
    return ids

In [ ]:
def load_stop_ids(file_path: str) -> Set[str]:
    '''Returns the set of stop_id from the file.'''
    ids: Set[str] = set()
    for r in read_dict_rows(file_path):
        tid = r.get('stop_id', '').strip()
        if tid:
            ids.add(tid)
    return ids

In [ ]:
def load_from_stop_ids(file_path: str) -> Set[str]:
    '''Returns the set of from_stop_id from the file.'''
    ids: Set[str] = set()
    for r in read_dict_rows(file_path):
        tid = r.get('from_stop_id', '').strip()
        if tid:
            ids.add(tid)
    return ids

In [ ]:
def load_to_stop_ids(file_path: str) -> Set[str]:
    '''Returns the set of to_stop_id from the file.'''
    ids: Set[str] = set()
    for r in read_dict_rows(file_path):
        tid = r.get('to_stop_id', '').strip()
        if tid:
            ids.add(tid)
    return ids

In [ ]:
def check_trip(trip_id: str, seqs_sorted: List[int]) -> List[str]:
    '''Checks that the stop_sequence values of a trip_id are consecutive. Returns
    error messages.'''
    msgs: List[str] = []
    last_seq = None
    for seq in seqs_sorted:
        if last_seq is not None and seq != last_seq + 1:
            msgs.append(
                f"Trip_id {trip_id}: stop_sequence does not increment by one ({last_seq} -> {seq})"
            )
        last_seq = seq
    return msgs

In [ ]:
def make_signature(item: Tuple[str, List[Tuple[int, str, str]]]) -> Tuple[str, Tuple[Tuple[int, str, str], ...]]:
    """
    Builds a canonical signature for a trip_id from its
    sequence of events (stop_sequence, arrival_time, departure_time),
    sorted by stop_sequence (int). Returns (trip_id, signature).
    """
    trip_id, rows = item
    normalized = tuple(sorted(rows, key=lambda x: x[0]))
    return trip_id, normalized

In [ ]:
# Shared helpers for pathway-based validations
PW_PAIR = re.compile(r'^PW\.(?P<a>[^_]+)_(?P<b>[^\s]+)$')

def iter_pathway_pairs(file_path: str) -> Iterable[Tuple[str, str, str]]:
    """Yields (pathway_id, a, b) for rows with pathway_id format 'PW.a_b'."""
    for r in read_dict_rows(file_path):
        pid = r.get('pathway_id', '').strip()
        if not pid:
            continue
        m = PW_PAIR.match(pid)
        if not m:
            continue
        yield pid, m.group('a'), m.group('b')

def load_transfer_pairs(file_path: str) -> Iterable[Tuple[str, str]]:
    """Generates (from_stop_id, to_stop_id) pairs from transfers.txt."""
    for r in read_dict_rows(file_path):
        a = r.get('from_stop_id', '').strip()
        b = r.get('to_stop_id', '').strip()
        if a or b:
            yield a, b

def load_stops_info(file_path: str) -> Dict[str, Tuple[str, str, str]]:
    """Returns stop_id -> (stop_name, stop_lat, stop_lon)."""
    out: Dict[str, Tuple[str, str, str]] = {}
    for r in read_dict_rows(file_path):
        sid = r.get('stop_id', '').strip()
        if not sid:
            continue
        out[sid] = (
            r.get('stop_name', '').strip(),
            r.get('stop_lat', '').strip(),
            r.get('stop_lon', '').strip(),
        )
    return out

def load_platforms_by_name(file_path: str) -> Dict[str, List[str]]:
    """Returns stop_name -> sorted unique list of platform stop_id (1.*)."""
    name_to_ones: Dict[str, List[str]] = {}
    for r in read_dict_rows(file_path):
        sid = r.get('stop_id', '').strip()
        sname = r.get('stop_name', '').strip()
        if not sid:
            continue
        if sid.startswith('1.'):
            name_to_ones.setdefault(sname, []).append(sid)
    for k in list(name_to_ones.keys()):
        name_to_ones[k] = sorted(set(name_to_ones[k]))
    return name_to_ones

def load_platform_pairs_present(file_path: str) -> Set[Tuple[str, str]]:
    """Returns sorted platform pairs (1.*, 1.*) that have a pathway in any direction."""
    pairs: Set[Tuple[str, str]] = set()
    for _, a, b in iter_pathway_pairs(file_path):
        if a.startswith('1.') and b.startswith('1.'):
            u, v = sorted((a, b))
            pairs.add((u, v))
    return pairs

#### Schedule case

In [ ]:
# Check in stop_times.txt (for stop_id == '1.433') that repeated times
# (departure_time) always correspond to different trip_id.
# Does not print all times, only a summary and problematic cases.

TARGET_STOP_ID='1.433'

def main():
    check_missing_files([STOP_TIMES_CLEANED_FILE])

    # time -> list of trip_ids (for this stop_id)
    time_to_trips: Dict[str, List[str]] = defaultdict(list)
    total_rows = 0
    matched_rows = 0

    for r in read_dict_rows(STOP_TIMES_CLEANED_FILE):
        total_rows += 1
        if r.get('stop_id', '') != TARGET_STOP_ID:
            continue
        dep = r.get('departure_time', '')
        tid = r.get('trip_id', '')
        if dep and tid:
            matched_rows += 1
            time_to_trips[dep].append(tid)

    unique_times = len(time_to_trips)
    repeated_times = {t: trips for t, trips in time_to_trips.items() if len(trips) >= 2}

    print(f"Total rows in stop_times_cleaned: {total_rows}")
    print(f"Target stop_id: {TARGET_STOP_ID}")
    print(f"Number of rows with departure_time and trip_id: {matched_rows}")
    print(f"\nUnique departure_time values: {unique_times}")
    print(f"Times appearing 2 or more times: {len(repeated_times)}")

    if not repeated_times:
        print("No repeated times for this stop_id.")
        return

    # Verify that for each repeated time, all trip_id are different
    violating = {}
    for t, trips in sorted(repeated_times.items()):
        counts = Counter(trips)
        # If any trip_id appears more than once for the same time, it's a violation
        bad = {trip_id: c for trip_id, c in counts.items() if c >= 2}
        if bad:
            violating[t] = bad

    if not violating:
        print("Correct: all repeated times correspond to different trip_id.")
    else:
        print("WARNING: repeated times found with the same trip_id (at least twice):")
        for t, bad_counts in violating.items():
            det = ", ".join([f"{trip_id} (x{c})" for trip_id, c in sorted(bad_counts.items())])
            print(f"- {t}: {det}")

# Execute
main()

In [ ]:
# List departure_time for a specific trip_id, showing stop_name instead of stop_id
# Target trip_id: '1.101.11426484'


TRIP_ID = '1.4.11633035'

def main():
    check_missing_files([STOP_TIMES_FILE, STOPS_FILE])

    stop_names = load_stop_names(STOPS_FILE)

    rows: List[Tuple[int, str, str]] = []  # (stop_sequence, departure_time, stop_name)
    total_rows = 0
    for r in read_dict_rows(STOP_TIMES_FILE):
        total_rows += 1
        if r.get('trip_id', '') != TRIP_ID:
            continue
        dep = r.get('departure_time', '')
        sid = r.get('stop_id', '')
        sname = stop_names.get(sid, '(no name)')
        seq_str = r.get('stop_sequence', '')
        try:
            seq = int(seq_str)
        except Exception:
            # If it cannot be converted, place it at the end
            seq = 10**9
        rows.append((seq, dep, sname))

    rows.sort(key=lambda x: (x[0], x[1]))

    print(f"Total rows in stop_times: {total_rows}")
    print(f"Target trip_id: {TRIP_ID}")
    print(f"Number of rows found for the trip: {len(rows)}")

    if not rows:
        print("(No rows found in stop_times for this trip_id)")
        return

    print("List (sorted by stop_sequence) of departure_time — stop_name:")
    for i, (seq, dep, sname) in enumerate(rows, 1):
        dep_show = dep if dep else '(no departure_time)'
        print(f"{i:02d}. {dep_show} — {sname}")

main()

In [ ]:
# List departure_time for a specific trip_id, showing stop_name instead of stop_id
# Target trip_id: '1.101.11426484'

TRIP_ID = '1.4.11633500'


def main():
    check_missing_files([STOP_TIMES_FILE, STOPS_FILE])

    stop_names = load_stop_names(STOPS_FILE)

    rows: List[Tuple[int, str, str]] = []  # (stop_sequence, departure_time, stop_name)
    total_rows = 0
    for r in read_dict_rows(STOP_TIMES_FILE):
        total_rows += 1
        if r.get('trip_id', '') != TRIP_ID:
            continue
        dep = r.get('departure_time', '')
        sid = r.get('stop_id', '')
        sname = stop_names.get(sid, '(no name)')
        seq_str = r.get('stop_sequence', '')
        try:
            seq = int(seq_str)
        except Exception:
            # If it cannot be converted, place it at the end
            seq = 10**9
        rows.append((seq, dep, sname))

    rows.sort(key=lambda x: (x[0], x[1]))

    print(f"Total rows in stop_times: {total_rows}")
    print(f"Target trip_id: {TRIP_ID}")
    print(f"Number of rows found for the trip: {len(rows)}")

    if not rows:
        print("(No rows found in stop_times for this trip_id)")
        return

    print("List (sorted by stop_sequence) of departure_time — stop_name:")
    for i, (seq, dep, sname) in enumerate(rows, 1):
        dep_show = dep if dep else '(no departure_time)'
        print(f"{i:02d}. {dep_show} — {sname}")

main()

In [ ]:
# List departure_time for a specific trip_id, showing stop_name instead of stop_id
# Target trip_id: '1.101.11426484'

TRIP_ID = '1.4.11644917'

def main():
    check_missing_files([STOP_TIMES_FILE, STOPS_FILE])

    stop_names = load_stop_names(STOPS_FILE)

    rows: List[Tuple[int, str, str]] = []  # (stop_sequence, departure_time, stop_name)
    total_rows = 0
    for r in read_dict_rows(STOP_TIMES_FILE):
        total_rows += 1
        if r.get('trip_id', '') != TRIP_ID:
            continue
        dep = r.get('departure_time', '')
        sid = r.get('stop_id', '')
        sname = stop_names.get(sid, '(no name)')
        seq_str = r.get('stop_sequence', '')
        try:
            seq = int(seq_str)
        except Exception:
            # If it cannot be converted, place it at the end
            seq = 10**9
        rows.append((seq, dep, sname))

    rows.sort(key=lambda x: (x[0], x[1]))

    print(f"Total rows in stop_times: {total_rows}")
    print(f"Target trip_id: {TRIP_ID}")
    print(f"Number of rows found for the trip: {len(rows)}")

    if not rows:
        print("(No rows found in stop_times for this trip_id)")
        return

    print("List (sorted by stop_sequence) of departure_time — stop_name:")
    for i, (seq, dep, sname) in enumerate(rows, 1):
        dep_show = dep if dep else '(no departure_time)'
        print(f"{i:02d}. {dep_show} — {sname}")

main()